# Example: GHZ state on the best qubits

We prepare a GHZ state with Qiskit and execute it on the optimal layout found by the core of IQM Qubit Selector
(`iqm.qubit_selector.core`), and on the naive layout `transpile_to_IQM` picks by default. The Hellinger fidelity to
the ideal GHZ distribution shows the difference.

The example runs on a crystal QPU (Garnet) first, and then on a Star QPU (Sirius).

In [ ]:
from collections.abc import Sequence

from iqm.iqm_client.transpile import transpile_remove_moves
from iqm.pulse import Circuit
from iqm.qiskit_iqm import IQMProvider, transpile_to_IQM
from iqm.qiskit_iqm.iqm_backend import IQMBackendBase
from iqm.qiskit_iqm.pulse_level_utils import qiskit_to_iqm
from iqm.qubit_selector.core import select_layout
from iqm.qubit_selector.core.ir import active_qubits, interactions_from_circuit
from qiskit import QuantumCircuit
from qiskit.quantum_info import hellinger_fidelity
from qiskit.visualization import plot_histogram

## Pinning the selected layout

Position `i` of a layout does not belong to virtual qubit `i`. It belongs to the `i`-th qubit of the converted
circuit in the selector's order: the qubits with two-qubit gates first, then the others, each group sorted by name
(as strings, so `QB10` comes before `QB2`). The converted circuit's qubits are the physical qubits Qiskit picked, so
`selected_initial_layout` follows each virtual qubit from the qubit Qiskit placed it on to the one the selector
chose. Pinning `[backend.qubit_name_to_index(q) for q in best_layout]` instead would scramble the qubits, and Qiskit
would add SWAPs to undo it.

In [ ]:
def selected_initial_layout(
    backend: IQMBackendBase, qc_transpiled: QuantumCircuit, circuit: Circuit, layout: Sequence[str]
) -> list[int]:
    """Return the ``initial_layout`` that pins the virtual qubits of ``qc_transpiled`` to ``layout``."""
    circuit = transpile_remove_moves(circuit)  # the selector reads a Star circuit without MOVEs
    core = sorted({qubit for pair in interactions_from_circuit(circuit) for qubit in pair})
    order = core + sorted(active_qubits(circuit) - set(core))
    moved_to = dict(zip(order, layout, strict=True))
    placed_at = qc_transpiled.layout.initial_index_layout(filter_ancillas=True)
    return [backend.qubit_name_to_index(moved_to[backend.index_to_qubit_name(p)]) for p in placed_at]

Switch on the logging of the selector to see what it does at each step: how many candidate layouts the enumerator
finds, whether the search budget cut it short, and the error budget of the best layout. The selector logs under
`iqm.qubit_selector`; the cell below shows its `INFO` messages in the notebook.

In [ ]:
import logging

selector_logger = logging.getLogger("iqm.qubit_selector")
selector_logger.setLevel(logging.INFO)  # logging.DEBUG for more detail, logging.WARNING for warnings only
if not selector_logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
    selector_logger.addHandler(handler)
selector_logger.propagate = True

# Crystal QPU

Set the `IQM_TOKEN` environment variable to your IQM Resonance API token before starting the notebook.
`IQMProvider` reads it from there, so the token never appears in the notebook.

In [ ]:
import os

token = "<YOUR_IQM_TOKEN_HERE>"
os.environ["IQM_TOKEN"] = token

iqm_server_url = "https://resonance.iqm.tech/"
quantum_computer = "garnet"  # a crystal QPU

backend = IQMProvider(iqm_server_url, quantum_computer=quantum_computer).get_backend()

## Define the GHZ state and find the optimal layout

Transpile the circuit to the backend with `transpile_to_IQM` and convert it with `qiskit_to_iqm`. The transpiled
circuit already sits on physical qubits that Qiskit picked, which is the naive layout run below; the selector treats
those names as placeholders and may move the circuit elsewhere. Given the client, the selector fetches the
architecture and metrics of the default calibration set itself.

In [ ]:
nqubits = 10
num_shots = 2000

qc = QuantumCircuit(nqubits)
qc.h(0)
for i in range(1, nqubits):
    qc.cx(i - 1, i)
qc.measure_all()

qc_transpiled = transpile_to_IQM(qc, backend, optimization_level=3)
converted, _ = qiskit_to_iqm(backend, qc_transpiled)
circuit = converted[0]

layouts = select_layout(circuit, client=backend.client, num_layouts=10)
print("Top 10 layouts and their errors:")
for layout, error in layouts:
    print(f"Layout: {layout}, Error: {error * 100:.2f}%")

Pin the best layout by turning it into the physical qubit index of each virtual qubit, which is the
`initial_layout` Qiskit expects.

In [ ]:
best_layout, _ = layouts[0]
best_layout_idx = selected_initial_layout(backend, qc_transpiled, circuit, best_layout)

print("Virtual qubit -> physical qubit:", {i: backend.index_to_qubit_name(q) for i, q in enumerate(best_layout_idx)})

## Execute on a naively chosen layout

In [ ]:
counts_naive = backend.run(qc_transpiled, shots=num_shots).result().get_counts()
plot_histogram(counts_naive)

## Execute on the optimal layout found by qubit selector

In [ ]:
qc_t = transpile_to_IQM(qc, backend, initial_layout=best_layout_idx, optimization_level=3)
counts_qubit_selector = backend.run(qc_t, shots=num_shots).result().get_counts()
plot_histogram(counts_qubit_selector)

## Evaluate the Hellinger fidelity

The optimal layout puts the gates on the qubits and couplers with the best calibration metrics, which shows up as a
higher GHZ fidelity than the naive run.

In [ ]:
counts_ideal = {"0" * nqubits: num_shots // 2, "1" * nqubits: num_shots // 2}
fidelity_naive = hellinger_fidelity(counts_ideal, counts_naive)
fidelity_qubit_selector = hellinger_fidelity(counts_ideal, counts_qubit_selector)

print(f"GHZ Hellinger fidelity naive: {fidelity_naive:.4f}")
print(f"GHZ Hellinger fidelity qubit selector: {fidelity_qubit_selector:.4f}")

# Star QPU

On Sirius every two-qubit gate goes through the computational resonator. The selector ranks the qubits with
fictional CZ fidelities that combine the resonator CZ and `MOVE` fidelities, and `transpile_to_IQM` adds the `MOVE`
gates on the pinned qubits. The steps are the same as above.

In [ ]:
star_backend = IQMProvider(iqm_server_url, quantum_computer="sirius").get_backend()

## Define the GHZ state and find the optimal layout

The 10-qubit GHZ state uses only part of Sirius, so there are layouts to choose from.

In [ ]:
star_nqubits = 10

star_qc = QuantumCircuit(star_nqubits)
star_qc.h(0)
for i in range(1, star_nqubits):
    star_qc.cx(i - 1, i)
star_qc.measure_all()

star_qc_transpiled = transpile_to_IQM(star_qc, star_backend, optimization_level=3)
star_converted, _ = qiskit_to_iqm(star_backend, star_qc_transpiled)
star_circuit = star_converted[0]

star_layouts = select_layout(star_circuit, client=star_backend.client, num_layouts=5)
print("Top 5 layouts and their errors:")
for layout, error in star_layouts:
    print(f"Layout: {layout}, Error: {error * 100:.2f}%")

In [ ]:
star_best_layout, _ = star_layouts[0]
star_best_layout_idx = selected_initial_layout(star_backend, star_qc_transpiled, star_circuit, star_best_layout)

print(
    "Virtual qubit -> physical qubit:",
    {i: star_backend.index_to_qubit_name(q) for i, q in enumerate(star_best_layout_idx)},
)

## Execute on a naively chosen layout

In [ ]:
star_counts_naive = star_backend.run(star_qc_transpiled, shots=num_shots).result().get_counts()
plot_histogram(star_counts_naive)

## Execute on the optimal layout found by qubit selector

In [ ]:
star_qc_t = transpile_to_IQM(star_qc, star_backend, initial_layout=star_best_layout_idx, optimization_level=3)
star_counts_qubit_selector = star_backend.run(star_qc_t, shots=num_shots).result().get_counts()
plot_histogram(star_counts_qubit_selector)

## Evaluate the Hellinger fidelity

In [ ]:
star_counts_ideal = {"0" * star_nqubits: num_shots // 2, "1" * star_nqubits: num_shots // 2}
star_fidelity_naive = hellinger_fidelity(star_counts_ideal, star_counts_naive)
star_fidelity_qubit_selector = hellinger_fidelity(star_counts_ideal, star_counts_qubit_selector)

print(f"GHZ Hellinger fidelity naive: {star_fidelity_naive:.4f}")
print(f"GHZ Hellinger fidelity qubit selector: {star_fidelity_qubit_selector:.4f}")